# Gap 5: Joint $(k, \lambda)$ Two-Dimensional Parameter Space

The graded SCN foundations paper defines TWO operators:
- **Threshold** $N_k$: hard-kill diagrams with nesting depth $\geq k$
- **Attenuation** $N_\lambda$: weight diagrams by $\lambda^{-\sigma}$

Previous gaps tested them separately. This gap tests the **combined** operator:

$$N_k \circ N_\lambda(x) = \begin{cases} \lambda^{-\sigma(x)} x & \sigma(x) < k \\ 0 & \sigma(x) \geq k \end{cases}$$

Does the 2D parameter space $(k, \lambda)$ reveal any region of genuine merit?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import math
import sys
sys.path.insert(0, '..')
from src.graded_scn import (
    ALPHA_S_MZ, M_Z, M_TAU,
    beta0_standard, beta0_attenuated, alpha_s_running_1loop,
    r_tau_coefficients, r_tau_tree,
    R_TAU_EXPERIMENT, R_TAU_UNCERTAINTY,
    ALPHA_OVER_PI, A_E_EXPERIMENT, A_E_UNCERTAINTY,
    QED_G2_COEFFICIENTS, compute_g2_standard,
)

# === Factorial Series: Combined (k, lambda) ===

def euler_borel(g):
    """Exact Borel sum of Euler series sum(-1)^n n! g^n."""
    return quad(lambda t: np.exp(-t)/(1+g*t), 0, np.inf)[0]

def euler_combined(g, k_thresh, lam, N=50):
    """Euler series under combined (k, lambda): terms with n < k weighted by lam^{-n}."""
    total = 0.0
        total += (-1)**n * math.factorial(n) * (g/lam)**n
        total += (-1)**n * np.math.factorial(n) * (g/lam)**n
    return total

g = 0.1
S_exact = euler_borel(g)

print(f"=== Euler Series: S(g={g}) ===")
print(f"Exact (Borel): S({g}) = {S_exact:.6f}")
print()

print(f"{'k':>4} {'lam':>6} {'S(g;k,lam)':>12} {'|error|':>12} {'S(g/lam)':>10} {'Match':>15}")
for k in [5, 10, 15, 20, 30]:
    for lam in [1.0, 1.5, 2.0, 3.0]:
        s_kl = euler_combined(g, k, lam)
        err = abs(s_kl - S_exact)
        s_rescaled = euler_borel(g/lam) if lam > 0 else np.nan
        note = ""
        if k >= 20 and abs(s_kl - s_rescaled) < 1e-4:
            note = "-> S(g/lam)"
        if err < 1e-4:
            note = "-> S(g) YES"
        print(f"{k:4d} {lam:6.1f} {s_kl:12.6f} {err:12.2e} {s_rescaled:10.6f} {note:>15}")
    print()

print("When k is large enough for convergence at the rescaled coupling:")
print("  S(g; k, lam) -> S(g/lam) -- the WRONG function.")
print("k just controls truncation; lambda just rescales.")
print("Neither transforms the factorial growth n! -> manageable.")

In [ ]:
# === R_tau with joint (k, lambda, f) ===

K = r_tau_coefficients()
nf = 3
b0_std = beta0_standard(nf)
alpha_s_mtau = alpha_s_running_1loop(M_TAU**2, nf, b0_std)
a_s = alpha_s_mtau / np.pi

def r_tau_joint(f, k_thresh, lam, beta0_att=False, max_order=4):
    """
    R_tau with combined nesting threshold + attenuation.
    
    Bubble chain in K_n has depth (n-1).
    depth >= k: killed.  depth < k: weighted by lam^{-depth}.
    Non-bubble part always at full weight.
    Optionally also attenuate beta_0.
    """
    if beta0_att:
        b0 = beta0_attenuated(nf, lam)
        as_eff = alpha_s_running_1loop(M_TAU**2, nf, b0)
        if np.isnan(as_eff):
            return np.nan
        a = as_eff / np.pi
    else:
        a = a_s
    
    delta = K[1] * a  # 1-loop: no bubble chains, depth 0
    for n in range(2, max_order + 1):
        depth = n - 1
        if depth >= k_thresh:
            k_eff = (1 - f) * K[n]  # bubble part killed
        else:
            k_eff = (1 - f) * K[n] + f * K[n] * lam**(-depth)
        delta += k_eff * a**n
    return r_tau_tree() * (1 + delta)

# Scan: fix f=0.5, vary (k, lambda)
f_fixed = 0.5
lam_vals = np.linspace(0.5, 5.0, 300)
k_vals = [1, 2, 3, 4, 100]

print(f"R_tau joint scan (f={f_fixed}): coefficients-only attenuation")
print(f"{'k':>4} {'lam_opt':>8} {'R_tau':>8} {'sigma':>8}")
for k in k_vals:
    r_arr = np.array([r_tau_joint(f_fixed, k, l) for l in lam_vals])
    resid = np.abs(r_arr - R_TAU_EXPERIMENT)
    best = np.argmin(resid)
    r_best = r_arr[best]
    sig = (r_best - R_TAU_EXPERIMENT) / R_TAU_UNCERTAINTY
    k_str = 'inf' if k >= 100 else str(k)
    print(f"{k_str:>4} {lam_vals[best]:8.2f} {r_best:8.4f} {sig:+8.1f}")

print()
print(f"Also with beta_0 attenuation (f={f_fixed}):")
for k in k_vals:
    r_arr = np.array([r_tau_joint(f_fixed, k, l, beta0_att=True) for l in lam_vals])
    valid = ~np.isnan(r_arr)
    if not valid.any():
        print(f"{'inf' if k>=100 else k:>4} {'N/A':>8} {'NaN':>8} {'---':>8}")
        continue
    resid = np.full_like(r_arr, 1e10)
    resid[valid] = np.abs(r_arr[valid] - R_TAU_EXPERIMENT)
    best = np.argmin(resid)
    r_best = r_arr[best]
    sig = (r_best - R_TAU_EXPERIMENT) / R_TAU_UNCERTAINTY
    k_str = 'inf' if k >= 100 else str(k)
    print(f"{k_str:>4} {lam_vals[best]:8.2f} {r_best:8.4f} {sig:+8.1f}")

In [ ]:
# Heatmap: |R_tau - exp| / sigma over (k, lambda)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
lam_fine = np.linspace(0.5, 5.0, 200)
k_fine = [1, 2, 3, 4]

for idx, (f_val, ax) in enumerate(zip([0.3, 0.5, 0.7], axes)):
    heatmap = np.zeros((len(k_fine), len(lam_fine)))
    for i, k in enumerate(k_fine):
        for j, lam in enumerate(lam_fine):
            r = r_tau_joint(f_val, k, lam)
            heatmap[i, j] = abs(r - R_TAU_EXPERIMENT) / R_TAU_UNCERTAINTY

    im = ax.imshow(heatmap, aspect='auto', origin='lower',
                   extent=[lam_fine[0], lam_fine[-1], 0.5, len(k_fine)+0.5],
                   vmin=0, vmax=20, cmap='RdYlGn_r')
    ax.set_xlabel('lambda')
    ax.set_ylabel('k_thresh')
    ax.set_yticks(range(1, len(k_fine)+1))
    ax.set_yticklabels(k_fine)
    ax.set_title(f'f={f_val}')
    plt.colorbar(im, ax=ax, label='|sigma|')

plt.suptitle('R_tau deviation from experiment in (k, lambda) space', fontsize=14)
plt.tight_layout()
plt.show()

print("Green regions: close to experiment. Red: far.")
print("Green regions exist for all f values -- but 3 free parameters")
print("(f, k, lambda) fitting 1 number (R_tau) is trivially achievable.")

In [ ]:
# === Structural impossibility analysis ===

print("=== Why the Combined Operator Cannot Help ===")
print()

print("1. FACTORIAL SERIES")
print("-" * 50)
print("Combined (k, lambda) on sum(-1)^n n! g^n:")
print("  = sum_{n<k} (-1)^n n! (g/lam)^n")
print("  = truncated S(g/lam)  [WRONG function, truncated]")
print("  lambda rescales, k truncates. Neither removes n! growth.")
print()

print("2. COUPLING RESCALING")
print("-" * 50)
print("For the loop-order scheme: combined just gives")
print("  truncated sum of K_n (a/lam)^n")
print("This is still a reparametrization, now truncated.")
print()

print("3. PER-DIAGRAM SCHEME")
print("-" * 50)
print("Combined (k, lam) on bubble chains:")
print("  depth < k: attenuated by lam^{-depth}")
print("  depth >= k: killed")
print("Three parameters (f, k, lam) for one observable (R_tau).")
print("Trivially fits by construction. No predictive content.")
print()

print("4. PARAMETER COUNT")
print("-" * 50)
print("Parameters: f (bubble fraction), k (threshold), lambda (attenuation)")
print("Observable: R_tau (one number)")
print("Degrees of freedom: 3 - 1 = 2 unconstrained")
print("Result: infinite family of solutions, zero predictions")
print()

print("5. THE FUNDAMENTAL ISSUE")
print("-" * 50)
print("Graded SCN reweights terms in a perturbative series.")
print("This cannot overcome:")
print("  (a) Factorial divergence: needs Borel or resurgence (structural transform)")
print("  (b) QED precision: any reweighting shifts predictions by >> exp. uncertainty")
print("  (c) QCD asymptotic freedom: the 'self-referential' gluon SE is essential")
print()
print("The mathematical axiom S in S => S = empty-set maps to 'kill self-energy'")
print("in physics. But self-energies are physically indispensable, not pathological.")

## Conclusions

The joint $(k, \lambda)$ parameter space was the most general possibility
within the graded SCN framework. Its exploration reveals:

1. **Factorial series**: Combined $(k, \lambda)$ gives a truncated version of $S(g/\lambda)$.
   Neither operator transforms the $n!$ growth — Borel resummation remains necessary.

2. **$R_\tau$**: With 3 free parameters $(f, k, \lambda)$ and 1 observable, fits are
   trivially achievable. Green regions in the heatmap are curve-fitting artifacts.

3. **Structural**: The combined operator is still a linear reweighting of perturbative
   terms. It cannot perform the nonlinear transformations (Borel, conformal mapping)
   that actually improve divergent series.

**Verdict**: The 2D parameter space offers no escape from the fundamental structural
limitations of multiplicative reweighting.